<br>

## **Import modules and functions**

---

In [1]:
import json
import pandas as pd

from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)

    return data

def postprocess(predictions, responses, questions, pred_length=100, response_pad_id=2):
    # pad 위치만 추출
    try:
        first_pad_index = responses.index(response_pad_id)
    except:
        first_pad_index = len(responses)

    if pred_length == 100:
        hme_question_prediction = predictions[first_pad_index-1]
        hme_question = questions[first_pad_index-1]
    elif pred_length == 99:
        hme_question_prediction = predictions[first_pad_index-2]
        hme_question = questions[first_pad_index-1]
    hme_question_response = responses[first_pad_index-1]

    pred_response = 1 if hme_question_prediction >= 0.5 else 0

    return hme_question_prediction, pred_response, hme_question_response, hme_question
    
def cal_metrics(y_true, y_pred, y_hat):
    confusion_mat = confusion_matrix(y_true, y_pred, labels=[0, 1])
    acc_per_class = confusion_mat.diagonal() / confusion_mat.sum(axis=1)
    
    # 각 metric 계산
    acc_wrong = acc_per_class[0]      # 학생이 틀린 문제에 대한 정확도
    acc_correct = acc_per_class[1]    # 학생이 맞춘 문제에 대한 정확도
    acc_micro = accuracy_score(y_true, y_pred)
    auc_micro = roc_auc_score(y_true, y_hat, average="micro", multi_class="ovr")

    return {"acc_wrong": float(acc_wrong), "acc_correct": float(acc_correct), "acc_micro": acc_micro, "auc_micro": float(auc_micro)}

<br>

## **Data Load**

---

In [2]:
question_meta_df = pd.read_csv("/home/jovyan/work/PICKT/data/Online/Total-Question_Meta.csv")
hme_question_meta = question_meta_df[question_meta_df['data_type']=='HME']

config = load_json("/home/jovyan/work/PICKT/data/Online/data_args.json")
id_question = {v:k for k,v in config['question2id'].items()}

hme_question_id_list = hme_question_meta['question_id'].tolist()

<br>

## **Prediction result Load**

In [3]:
model_name = 'pickt'
task = "pred"
prediction_result = load_json(f"/home/jovyan/work/repo/models/Offline/RQ2/{model_name}_{task}/predict_outputs.json")

if model_name in ["dkt", "dkvmn", "sakt", "gkt"]:
    hme_25_topic_list = [config['question2concept'][str(quizcode)] for quizcode in hme_question_id_list]
    hme_25_topic_set = set(hme_25_topic_list)
    
    topic_name_list = [config['concept2id'][topic_name] for topic_name in hme_25_topic_set]
    len(topic_name_list) == len(hme_25_topic_set)
elif model_name in ["pickt", "dtransformer", "saint", "akt"]:
    pass
else:
    print('else')

print(model_name)
print(prediction_result.keys())

pickt
dict_keys(['predictions', 'responses', 'questions'])


<br>

## Eval Performance in Cold-Start scenario

In [4]:
count = 0
pred_score_list, pred_resp_list, label_list = list(), list(), list()
for predictions, responses, questions in zip(prediction_result['predictions'], prediction_result['responses'], prediction_result['questions']):
    pred_score, pred_response, label, hme_question = postprocess(predictions, responses, questions, pred_length=len(predictions))

    if model_name in ["dkt", "dkvmn", "sakt", "gkt"]:
        if hme_question in topic_name_list:
            pred_score_list.append(pred_score)
            pred_resp_list.append(pred_response)
            label_list.append(label)
        else:
            count += 1
    elif model_name in ["pickt", "dtransformer", "saint", "akt"]:
        if int(id_question.get(hme_question)) in hme_question_id_list:
            id_question.get(hme_question)
            pred_score_list.append(pred_score)
            pred_resp_list.append(pred_response)
            label_list.append(label)
        else:
            count += 1
    else:
        break

print(count)
print(len(predictions), len(responses), len(questions))
cal_metrics(label_list, pred_resp_list, pred_score_list)

300
100 100 100


{'acc_wrong': 0.4091580502215657,
 'acc_correct': 0.9522191329420245,
 'acc_micro': 0.7741176470588236,
 'auc_micro': 0.8339576538932307}